# AlgoChowk Quantitative Event Study

Frozen reproducible presentation layer separating full-sample analysis from the event study.

In [1]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.data_loader import load_nifty_data
from src.events import detect_events, calculate_forward_returns
from src.full_sample import calculate_full_sample_forward_returns, add_return_buckets, summarize_return_buckets, summarize_populations
HORIZONS = (1, 3, 5, 10)
SPLIT = pd.Timestamp('2020-01-01')
df = load_nifty_data(ROOT / 'data/raw/nifty50.csv').sort_values('Date').reset_index(drop=True)
print('Total observations:', len(df))
print('Date range:', df.Date.min().date(), 'to', df.Date.max().date())
print('Development observations:', (df.Date < SPLIT).sum())
print('OOS observations:', (df.Date >= SPLIT).sum())
print('Missing values:', df.isna().sum().sum())

Total observations: 4585
Date range: 2007-09-17 to 2026-05-29
Development observations: 3001
OOS observations: 1584
Missing values: 0


## Full-sample forward-return analysis

In [2]:
full_sample = add_return_buckets(calculate_full_sample_forward_returns(df, HORIZONS))
print('Full-sample rows:', len(full_sample))
display(summarize_return_buckets(full_sample, HORIZONS))
events = calculate_forward_returns(df, detect_events(df, threshold=-0.03, overlap_window=5), HORIZONS)
display(summarize_populations(full_sample, events, HORIZONS))

Full-sample rows: 4585


,return_bucket,n,mean_return,mean_forward_return_1,median_forward_return_1,win_rate_1,mean_forward_return_3,median_forward_return_3,win_rate_3,mean_forward_return_5,median_forward_return_5,win_rate_5,mean_forward_return_10,median_forward_return_10,win_rate_10
0,return <= -3%,79,-0.044803,-0.000498,0.000188,0.506329,0.001771,0.006782,0.556962,0.004228,0.013597,0.569620,0.002978,0.015440,0.582278
1,-3% < return <= -2%,120,-0.023740,-0.000318,-0.000872,0.475000,0.000764,0.003914,0.541667,-0.000193,-0.000735,0.491667,0.005325,0.009748,0.533333
2,-2% < return <= -1%,445,-0.013864,-0.000191,-0.000017,0.497748,0.002090,0.003171,0.554054,0.004514,0.004850,0.569820,0.004580,0.004036,0.560811
3,-1% < return <= 0%,1510,-0.004085,-0.000904,-0.000600,0.465563,-0.000260,0.000059,0.501326,0.000506,0.001706,0.533820,0.002418,0.004183,0.550831
4,return > 0%,2430,0.008548,-0.000319,-0.000448,0.476132,0.000395,0.001235,0.535802,0.001070,0.002254,0.544893,0.003611,0.005467,0.569662


,population,n,mean_return,mean_forward_return_1,mean_forward_return_3,mean_forward_return_5,mean_forward_return_10
0,All observations,4585,0.000446,-0.000500,0.000389,0.001261,0.003374
1,Extreme negative events <= -3%,53,-0.040556,-0.000290,0.001727,0.007423,0.003312
2,Non-event observations,4532,0.000926,-0.000502,0.000373,0.001189,0.003374


## Event definition and event-study results

In [3]:
events['period'] = events['Date'].lt(SPLIT).map({True: 'development', False: 'oos'})
print('threshold = -0.03; overlap_window = 5')
print('Total events:', len(events))
print('Development events:', (events.period == 'development').sum())
print('OOS events:', (events.period == 'oos').sum())
display(pd.read_csv(ROOT / 'results/core_results.csv'))

threshold = -0.03; overlap_window = 5
Total events: 53
Development events: 38
OOS events: 15


,period,horizon,event_n,baseline_n,event_mean,baseline_mean,mean_difference,event_median,baseline_median,event_win_rate,baseline_win_rate
0,development,1,38,3001,-0.002715,-0.000484,-0.002231,-0.002076,-0.000588,0.421053,0.469510
1,development,3,38,3001,0.001889,0.000370,0.001519,0.005385,0.001007,0.578947,0.523159
2,development,5,38,3001,0.008975,0.001171,0.007805,0.013162,0.002257,0.605263,0.544152
3,development,10,38,3001,0.004538,0.003103,0.001435,0.007283,0.004810,0.526316,0.557148
4,oos,1,15,1583,0.005853,-0.000529,0.006382,0.008019,-0.000177,0.800000,0.486418
5,oos,3,15,1581,0.001316,0.000425,0.000891,0.014141,0.001216,0.600000,0.533839
6,oos,5,15,1579,0.003491,0.001432,0.002059,0.016024,0.002319,0.600000,0.540215
7,oos,10,15,1574,0.000207,0.003890,-0.003683,0.023626,0.005100,0.733333,0.571156


## Threshold sensitivity, bootstrap uncertainty, and regime robustness

In [4]:
display(pd.read_csv(ROOT / 'results/event_diagnostics.csv'))
display(pd.read_csv(ROOT / 'results/bootstrap_results.csv'))
regimes = pd.read_csv(ROOT / 'results/regime_results.csv')
display(regimes[regimes.status == 'inferential'])
print('Cells below event_n=10 are descriptive_only.')

,threshold,raw_qualifying,overlap_filtered,development_events,oos_events
0,-0.020,199,123,92,31
1,-0.025,117,75,54,21
2,-0.030,79,53,38,15
3,-0.035,50,35,26,9
4,-0.040,37,26,20,6
5,-0.050,22,15,11,4


,period,horizon,event_n,baseline_n,observed_difference,bootstrap_se,ci_lower,ci_upper,p_value
0,development,1,38.0,3001.0,-0.002231,0.004445,-0.011032,0.006465,0.6615
1,development,3,38.0,3001.0,0.001519,0.007749,-0.014718,0.015985,0.8537
2,development,5,38.0,3001.0,0.007805,0.007995,-0.008118,0.023675,0.5190
3,development,10,38.0,3001.0,0.001435,0.012470,-0.023722,0.025476,0.9102
4,oos,1,15.0,1583.0,0.006382,0.003718,-0.001150,0.013389,0.5238
5,oos,3,15.0,1581.0,0.000891,0.008058,-0.015166,0.016038,0.9103
6,oos,5,15.0,1579.0,0.002059,0.011489,-0.022820,0.022208,0.8696
7,oos,10,15.0,1574.0,-0.003683,0.024799,-0.056682,0.039457,0.8835


,period,regime,horizon,event_n,baseline_n,event_mean,baseline_mean,mean_difference,status,partition
0,development,downtrend,1,31,1231,-0.004898,-0.000843,-0.004056,inferential,trend
1,development,downtrend,3,31,1231,-0.006096,0.000025,-0.006121,inferential,trend
2,development,downtrend,5,31,1231,0.000880,0.001172,-0.000292,inferential,trend
3,development,downtrend,10,31,1231,-0.005445,0.003070,-0.008515,inferential,trend
8,oos,downtrend,1,12,613,0.007235,-0.000428,0.007663,inferential,trend
9,oos,downtrend,3,12,613,0.000162,0.000386,-0.000223,inferential,trend
10,oos,downtrend,5,12,612,-0.000708,0.001721,-0.002428,inferential,trend
11,oos,downtrend,10,12,607,-0.001784,0.003941,-0.005726,inferential,trend
16,development,high,1,12,816,-0.001027,-0.000927,-0.000100,inferential,volatility
17,development,high,3,12,816,0.002128,-0.000679,0.002807,inferential,volatility


Cells below event_n=10 are descriptive_only.


## Final hypothesis assessment

The full-sample analysis describes conditional forward returns across approximately 4,500 observations. The event study remains a separate 53-event analysis, split into 38 development and 15 OOS events. Positive point estimates occur at some horizons, but bootstrap confidence intervals cross zero, including OOS. The evidence does not establish a statistically robust positive abnormal-return effect and does not imply a tradable strategy.